In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to gold layer
con = duckdb.connect('../data/gold/gold.duckdb')

# Style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("Connected ✓")

In [ ]:
    # ── KPI 1: Overall Default Rate ──────────────────────────────────────────────
kpi = con.execute("""
        SELECT
            COUNT(*) AS total_loans,
            SUM(default_flag) AS total_defaults,
            ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct,
            ROUND(SUM(funded_amnt) / 1e9, 2) AS total_funded_bn,
            ROUND(AVG(funded_amnt), 2) AS avg_loan_amount,
            ROUND(AVG(int_rate), 2) AS avg_interest_rate
        FROM fact_loans
    """).fetchdf()

print("=== OVERALL KPIs ===")
print(kpi.to_string(index=False))

In [ ]:
# ── KPI 2: Default Rate by Sub Grade ──────────────────────────────────────────────
by_grade = con.execute("""
    SELECT
        dr.sub_grade,
        COUNT(*) AS total_loans,
        ROUND(AVG(f.default_flag) * 100, 2) AS default_rate_pct,
        ROUND(AVG(f.int_rate), 2) AS avg_int_rate
    FROM fact_loans f
    JOIN dim_risk dr ON f.risk_key = dr.risk_key
    GROUP BY dr.sub_grade
    ORDER BY dr.sub_grade
""").fetchdf()

# Plot
fig, ax = plt.subplots()
ax.bar(by_grade['sub_grade'], by_grade['default_rate_pct'], color='steelblue')
ax.set_xlabel('Sub Grade')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Sub Grade')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(by_grade.to_string(index=False))

In [ ]:
# ── KPI 3: Default Rate by Purpose ──────────────────────────────────────────────
by_purpose = con.execute("""
    SELECT
        dp.purpose,
        COUNT(*) AS total_loans,
        ROUND(AVG(f.default_flag) * 100, 2) AS default_rate_pct,
        ROUND(AVG(f.funded_amnt), 2) AS avg_loan_amount
    FROM fact_loans f
    JOIN dim_purpose dp ON f.purpose_key = dp.purpose_key
    GROUP BY dp.purpose
    ORDER BY default_rate_pct DESC
""").fetchdf()

# Plot
fig, ax = plt.subplots()
colors = ['crimson' if x > 21.21 else 'steelblue' for x in by_purpose['default_rate_pct']]
ax.barh(by_purpose['purpose'], by_purpose['default_rate_pct'], color=colors)
ax.axvline(x=21.21, color='black', linestyle='--', label='Avg 21.21%')
ax.set_xlabel('Default Rate (%)')
ax.set_title('Default Rate by Loan Purpose')
ax.legend()
plt.tight_layout()
plt.show()

print(by_purpose.to_string(index=False))

In [ ]:
# ── KPI 4: Default Rate by State ──────────────────────────────────────────────
by_state = con.execute("""
    SELECT
        addr_state,
        COUNT(*) AS total_loans,
        ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct
    FROM fact_loans
    GROUP BY addr_state
    HAVING COUNT(0) > 100
    ORDER BY default_rate_pct DESC
    LIMIT 15
""").fetchdf()

# Plot
fig, ax = plt.subplots()
ax.barh(by_state['addr_state'], by_state['default_rate_pct'], color='steelblue')
ax.axvline(x=21.21, color='black', linestyle='--', label='Avg 21.21%')
ax.set_xlabel('Default Rate (%)')
ax.set_title('Top 15 States by Default Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── KPI 5: Default Rate Over Time ──────────────────────────────────────────────
by_time = con.execute("""
    SELECT
        dd.year,
        dd.quarter,
        COUNT(*) AS total_loans,
        ROUND(AVG(f.default_flag) * 100, 2) AS default_rate_pct
    FROM fact_loans f
    JOIN dim_date dd ON f.date_key = dd.date_key
    GROUP BY dd.year, dd.quarter
    ORDER BY dd.year, dd.quarter
""").fetchdf()

by_time['period'] = by_time['year'].astype(str) + '-Q' + by_time['quarter'].astype(str)

# Plot
fig, ax = plt.subplots()
ax.plot(by_time['period'], by_time['default_rate_pct'], marker='o', color='steelblue')
ax.axhline(y=21.21, color='black', linestyle='--', label='Avg 21.21%')
ax.set_xlabel('Period')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate Over Time')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── KPI 6: Default Rate by Income Category ──────────────────────────────────────────────
by_income = con.execute("""
    SELECT
        CASE 
            WHEN annual_inc < 40000 THEN '1. Low (<40k)'
            WHEN annual_inc < 80000 THEN '2. Medium (40k-80k)'
            WHEN annual_inc < 120000 THEN '3. High (80k-120k)'
            ELSE '4. Very High (>120k)'
        END AS income_category,
        COUNT(*) AS total_loans,
        ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct,
        ROUND(AVG(dti), 2) AS avg_dti
    FROM fact_loans
    GROUP BY income_category
    ORDER BY income_category
""").fetchdf()

fig, ax = plt.subplots()
ax.bar(by_income['income_category'], by_income['default_rate_pct'], color='steelblue')
ax.axhline(y=21.21, color='black', linestyle='--', label='Avg 21.21%')
ax.set_xlabel('Income Category')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Income Category')
ax.legend()
plt.tight_layout()
plt.show()

print(by_income.to_string(index=False))

In [ ]:
con.close()
print("Analysis complete ✓")

In [ ]:
con = duckdb.connect('../data/gold/gold.duckdb')
check = con.execute("""
    SELECT
        dp.purpose,
        COUNT(*) AS total_loans,
        ROUND(AVG(f.default_flag) * 100, 2) AS default_rate_pct
    FROM fact_loans f
    JOIN dim_purpose dp ON f.purpose_key = dp.purpose_key
    GROUP BY dp.purpose
    ORDER BY default_rate_pct DESC
""").fetchdf()
print(check)